# MiniProyecto Deep Learning
## Elaborado por Fabián Calvo Castillo - Florencia Pellegrini

In [ ]:
#%pip install feedparser trafilatura
#%pip install google-generativeai

In [ ]:
#%pip install python-dotenv

In [ ]:
#%pip install edge-tts
#%pip install nest_asyncio
#%pip install openai-whisper

In [ ]:
#%pip install moviepy==1.0.3
#%pip install Pillow

In [1]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)
from news_fetcher import get_all_news
import google.generativeai as genai
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin
import t2s
importlib.reload(t2s)
from t2s import generar_audio

C:\Users\fcalv\AppData\Local\Temp\ipykernel_15432\2596570605.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Paso 1: Obtener noticias

In [4]:
import importlib
import news_fetcher
importlib.reload(news_fetcher)

from news_fetcher import get_all_news
articulos = get_all_news(num_per_source=3)

RECOPILANDO NOTICIAS
\n-> Leyendo RSS de todas las fuentes...
  [ABC] 3 articulos en RSS
  [ElDiario] 3 articulos en RSS
  [Europa Press] 3 articulos en RSS
  [La Vanguardia] 3 articulos en RSS
  inutos] 3 articulos en RSS
  [Antena 3] 3 articulos en RSS
  [RTVE] 3 articulos en RSS
  [El Pais] 3 articulos en RSS
  [El Mundo] 3 articulos en RSS
\n-> Extrayendo texto completo (27 articulos)...
  Procesando: La EMT modifica rutas y suspende líneas por la celebrac...
    -> Texto completo (2716 chars)
  Procesando: La consellera Marián Cano, el empresario José González ...
    -> Texto completo (2997 chars)
  Procesando: El día a día de Kairu, el 'bebé' rinoceronte integrado ...
    -> Texto completo (3787 chars)
  Procesando: La fundación financiada por Vox declaró 630.000 euros e...
    -> Texto completo (6212 chars)
  Procesando: Vox se querella contra Joan Baldoví por sus críticas a ...
    -> Texto completo (1976 chars)
  Procesando: La Diputación de Valencia ficha en comisión de serv

Con esta función se guarda una variable articulos, que es una lista de diccionarios Python, uno por cada artículo. 
Así luego el LLM recibirá todos los textos completos agrupados por tema y devolverá un único resumen redactado de forma neutral, sin subjetivismo político. 

In [5]:
# Comprobación

for i, art in enumerate(articulos):
    print(f"\n{'='*40}")
    print(f"Artículo {i+1}")
    print(f"Fuente:  {art['fuente']}")
    print(f"Título:  {art['titulo']}")
    print(f"Origen:  {art['texto_origen']}")
    print(f"Chars:   {len(art['texto_completo'])}")
    print(f"Preview: {art['texto_completo'][:200]}...")


Artículo 1
Fuente:  ABC
Título:  La EMT modifica rutas y suspende líneas por la celebración de la Semana Santa en Madrid
Origen:  completo
Chars:   2716
Preview: Madrid se prepara para la celebración de la Semana Santa con un amplio dispositivo de movilidad que implicará cambios significativos en el transporte público. La Empresa Municipal de Transportes (EMT ...

Artículo 2
Fuente:  ABC
Título:  La consellera Marián Cano, el empresario José González y el torero Vicente Barrera, granaderos de honor 2026
Origen:  completo
Chars:   2997
Preview: Valencia
La Corporació de Granaders de la Verge de la Parroquia de los Ángeles del Cabanyal de Valencia ha nombrado a los Granaderos de Honor de 2026 cuyos cargos han recaído en a la Consellera de Ind...

Artículo 3
Fuente:  ABC
Título:  El día a día de Kairu, el 'bebé' rinoceronte integrado en la sabana de Bioparc Valencia
Origen:  completo
Chars:   3787
Preview: Valencia
«Mira cómo corre detrás de las cebras» o «verlo es una auténtica maravill

## Paso 2: Generación del resumen

Se toma el diccionario generado con las noticias que se recopilaron en distintos medios, y se genera el resumen haciendo uso de la API de Gemini

In [6]:
import google.generativeai as genai
import importlib
import summarizer
importlib.reload(summarizer)
import os
from dotenv import load_dotenv
from summarizer import resumir_noticias, guardar_boletin

load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
#GEMINI_API_KEY = "..."

genai.configure(api_key=GEMINI_API_KEY)
mi_modelo_gemini = genai.GenerativeModel("gemini-2.5-flash")

# Pipeline de resumen
boletin_final, resumenes = resumir_noticias(articulos, mi_modelo_gemini)

if boletin_final:
    print("\n" + "*"*50)
    print("BOLETÍN DE NOTICIAS")
    print("*"*50 + "\n")
    print(boletin_final)
    guardar_boletin(boletin_final) 


RESUMIENDO 27 ARTÍCULOS

-> Agrupando artículos por temas...
   6 temas identificados:
   - Conflicto en Oriente Medio y sus implicaciones (10 fuentes: Europa Press, Europa Press, La Vanguardia, La Vanguardia, La Vanguardia, Antena 3, RTVE, El Pais, El Pais, El Pais)
   - Economía (Euríbor, Bolsa y Petróleo) (3 fuentes: Europa Press, 20minutos, 20minutos)
   - Actualidad política española (general/partidos) (4 fuentes: ElDiario, ElDiario, 20minutos, El Mundo)
   - Política y elecciones en Andalucía (2 fuentes: El Mundo, El Mundo)
   - Noticias de la Comunidad Valenciana (3 fuentes: ABC, ABC, ElDiario)
   - Eutanasia y Derechos Humanos (1 fuentes: RTVE)

-> Generando resúmenes por tema...
   [1/6] Conflicto en Oriente Medio y sus implicaciones (10 fuentes)...
      OK (1151 chars)
      Error: type object 'datetime.time' has no attribute 'sleep'
   [2/6] Economía (Euríbor, Bolsa y Petróleo) (3 fuentes)...
      OK (913 chars)
      Error: type object 'datetime.time' has no attribute 's

## Paso 3: Síntesis de voz (T2S)

In [8]:
import importlib
import t2s
importlib.reload(t2s)
from t2s import generar_audio

ruta_audio, ruta_subtitulos = generar_audio(boletin_final)

Generando audio...
  Voz:         es-ES-AlvaroNeural
  Chars:       6363
  Audio:       audios/audio_2026-03-25_09-37-36.mp3
Audio generado! (2.22 MB)
  Generando subtítulos con Whisper...


c:\Users\flore\anaconda3\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


  Subtítulos generados: audios/subtitulos_2026-03-25_09-37-36.vtt


In [9]:
# Verificar que el VTT tiene contenido
with open(ruta_subtitulos, "r", encoding="utf-8") as f:
    contenido = f.read()
print(f"VTT tiene {len(contenido)} chars")
print(contenido[:300])

VTT tiene 15193 chars
WEBVTT

00:00:00.000 --> 00:00:01.240
Muy buenas tardes.

00:00:02.240 --> 00:00:03.879
Bienvenidos a nuestro informativo.

00:00:04.879 --> 00:00:06.440
Comenzamos este repaso por

00:00:06.440 --> 00:00:07.799
las principales noticias del

00:00:07.799 --> 00:00:09.019
día, con la mirada

00:00:09


## Paso 4: Generación del video resumen 

In [10]:
import subprocess
resultado = subprocess.run(["where", "magick"], capture_output=True, text=True)
print(resultado.stdout)

C:\Program Files\ImageMagick-7.1.2-Q16-HDRI\magick.exe



In [11]:
import importlib
import video_maker
importlib.reload(video_maker)
from video_maker import generar_video

PEXELS_API_KEY = os.getenv('PEXELS_API_KEY')
#PEXELS_API_KEY = "..."

ruta_video = generar_video(
    ruta_audio, ruta_subtitulos, resumenes,
    PEXELS_API_KEY,
    modelo_ia=mi_modelo_gemini  # para mejorar las queries de imagen
)


GENERANDO VÍDEO

1. Cargando audio...
   Duración total:    387.8s
   Temas:             6
   Duración por tema: 64.6s

2. Buscando imágenes en Pexels...
  Tema: Conflicto en Oriente Medio y sus implicaciones...
    Query imagen: 'Conflict
Aftermath'
    Keywords: 'conflict aftermath'
    Imagen guardada: imagenes/img_3282972612071549955.jpg
  Tema: Economía (Euríbor, Bolsa y Petróleo)...
    Query imagen: 'Market Volatility'
    Keywords: 'market volatility'
    Imagen guardada: imagenes/img_8846086146050120290.jpg
  Tema: Actualidad política española (general/partidos)...
    Query imagen: 'Spain Politics'
    Keywords: 'spain politics'
    Imagen guardada: imagenes/img_8863019462695661148.jpg
  Tema: Política y elecciones en Andalucía...
    Query imagen: 'Andalusia
Rally'
    Keywords: 'andalusia rally'
    Imagen guardada: imagenes/img_7436943988949425405.jpg
  Tema: Noticias de la Comunidad Valenciana...
    Query imagen: 'Mediterranean News'
    Keywords: 'mediterranean news'
 